# Carbon Mass of Plastic Compared with Life
> **Author:** Dennis Zhu\
> **Date Started:** November 4th, 2026\
> **Input:** `input/product-to-year.csv` from the sister notebook, `plastic-production_data-wrangling.ipynb`. Biomass from [Bar-On, 2018](https://www.pnas.org/doi/10.1073/pnas.1711842115).

---

## 1. Objective

Convert plastic production into carbon mass so it can be compared with the carbon in living things (Fig. 1A). This puts section 2.3 of the sister notebook into code, moves it to 2026, and adds plastic that has not been incinerated.

## 2. Experiment
### 2.1. Carbon Fraction of Plastic
---
Same method as the sister notebook: carbon mass over repeat unit mass for each polymer, weighted by its share of production ([Geyer, 2017](https://www.science.org/doi/10.1126/sciadv.1700782)). PUR and other polymers (13.1% of production) are left out, and the remaining shares are renormalized.

In [ ]:
# ———— ATOMIC MASSES ———— #
MW <- c(C = 12.011, H = 1.008, O = 15.999, Cl = 35.45)

# ———— REPEAT UNITS AND PRODUCTION SHARES (%) ———— #
polymers <- data.frame(
  polymer = c("PE", "PP", "PVC", "PET", "PS"),
  share   = c(36.3, 21.0, 11.8, 10.2, 7.6),
  C       = c(2, 3, 2, 10, 8),
  H       = c(4, 6, 3, 8, 8),
  O       = c(0, 0, 0, 4, 0),
  Cl      = c(0, 0, 1, 0, 0)
)

polymers$carbon_fraction <- with(polymers,
  C * MW[["C"]] / (C * MW[["C"]] + H * MW[["H"]] + O * MW[["O"]] + Cl * MW[["Cl"]]))

# weighted.mean renormalizes the shares to the five polymers
CARBON_FRACTION <- weighted.mean(polymers$carbon_fraction, polymers$share)

print(polymers[, c("polymer", "share", "carbon_fraction")], digits = 3)
cat(sprintf("\nWeighted carbon fraction: %.4f\n", CARBON_FRACTION))

### 2.2. Plastic Mass by Fate
---
Cumulative totals since 1950 come from the CSV. Two definitions of plastic that still exists:

- **Remaining** = produced − recycled − incinerated. This is the sister notebook's definition (landfill, environment and in use).
- **Not incinerated** = produced − incinerated. Recycled plastic is still plastic; only incineration turns its carbon into CO₂. This is the better match for the plastic that exists today.

In [ ]:
YEAR <- 2026
plastic <- read.csv("input/product-to-year.csv")

plastic_by_fate <- function(year) {
  r <- plastic[plastic$Year == year, ]
  data.frame(
    quantity   = c("Produced", "Recycled", "Incinerated", "Remaining", "Not incinerated"),
    plastic_Gt = c(r$Total.Produced,
                   r$Total.Recycled,
                   r$Total.Incinerated,
                   r$Total.Produced - r$Total.Recycled - r$Total.Incinerated,
                   r$Total.Produced - r$Total.Incinerated)
  )
}

plastic_by_fate(YEAR)

### 2.3. Carbon Mass and Comparison with Life
---
Plastic carbon is plastic mass times the carbon fraction. Life values are Bar-On 2018, in Gt C. Animals are 2 Gt C, which replaces the 2.589 from Visual Capitalist used in the sister notebook.

As a check, 2024 should give the sister notebook's 9.331 Gt C produced and 6.347 Gt C remaining. Its numbers are rounded slightly differently, so they match to within 0.005.

In [ ]:
# ———— CARBON IN LIVING THINGS (Gt C) ———— #
life <- data.frame(
  group      = c("Plants", "Bacteria", "Fungi", "Archaea", "Protists", "Animals"),
  carbon_GtC = c(450, 70, 12, 7, 4, 2)
)
ANIMALS <- life$carbon_GtC[life$group == "Animals"]

carbon_table <- function(year) {
  t <- plastic_by_fate(year)
  t$carbon_GtC <- t$plastic_Gt * CARBON_FRACTION
  t$x_animals  <- t$carbon_GtC / ANIMALS
  t
}

# ———— CHECK AGAINST THE SISTER NOTEBOOK (2024) ———— #
check <- carbon_table(2024)
stopifnot(abs(check$carbon_GtC[check$quantity == "Produced"]  - 9.331) < 0.005,
          abs(check$carbon_GtC[check$quantity == "Remaining"] - 6.347) < 0.005)

# ———— RESULT ———— #
result <- carbon_table(YEAR)
print(result, digits = 3, row.names = FALSE)

## 3. Results
---
Plastic in 2026, with plastic mass in Gt and carbon in Gt C:

| quantity        | plastic (Gt) | carbon (Gt C) | × animals |
| --------------- | ------------ | ------------- | --------- |
| Produced        | 13.19        | 10.17         | 5.1       |
| Recycled        | 1.82         | 1.41          | 0.7       |
| Incinerated     | 2.66         | 2.05          | 1.0       |
| Remaining       | 8.70         | 6.71          | 3.4       |
| Not incinerated | 10.53        | 8.11          | 4.1       |

Carbon fraction: 0.771. Plastic that has not been incinerated holds about 4 times the carbon in all animals. If recycled plastic is also excluded, the figure is 3.4 times.

Limits:
- The recycling and incineration rates follow the sister notebook's assumptions: a 1 year product life and Geyer's linear rate trend.
- "Remaining" does not separate landfill, the environment and plastic in use.
- The carbon fraction leaves out PUR and other polymers. The biomass estimates are uncertain, up to about 10-fold for bacteria and archaea.